# Generating Indian Names using Sequence Models

## Loading Required Libraries and Data

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import CrossEntropyLoss
import torch.nn.functional as F
import random

In [5]:
# Load data
with open("data/TrainingNames.txt", 'r') as f:
    names = [line.strip().lower() for line in f if line.strip()]

## Preprocessing Data

In [7]:
# Vocabulary
chars = sorted(set("".join(names)))
vocab = ["<PAD>", "<SOS>", "<EOS>"] + chars  # add start , end and pad characters to the vocab

char2idx = {c:i for i,c in enumerate(vocab)}  # create a map for character to index conversion
idx2char = {i:c for c,i in char2idx.items()}  # create a map for index to character conversion

vocab_size = len(vocab)

In [8]:
# Encode each name as start character + name + end character in indices 
def encode(name):
    return [char2idx["<SOS>"]] + [char2idx[c] for c in name] + [char2idx["<EOS>"]] 

encoded = [encode(n) for n in names]

In [9]:
# Padding to maintain uniform length 

max_len = max(len(seq) for seq in encoded)

def pad(seq):
    return seq + [char2idx["<PAD>"]] * (max_len - len(seq))

X = []
Y = []

for seq in encoded:
    X.append(pad(seq[:-1]))
    Y.append(pad(seq[1:]))

X = torch.tensor(X)
Y = torch.tensor(Y)

## Modelling and Training

### Define Hyperparameters and Functions

In [12]:
# train function 
def train(model, X, Y, epochs=150, LR = 0.001):
    optimizer = optim.Adam(model.parameters(), lr = LR) # define optimizer
    loss_fn = CrossEntropyLoss(ignore_index=char2idx["<PAD>"]) # define loss and ignore padding characters for loss computation

    for epoch in range(epochs):
        model.train()

        out = model(X)
        loss = loss_fn(out.view(-1, vocab_size), Y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


# define funtion to count the learnable parameters of a model

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### RNN

In [14]:
# define RNN architecture

class RNN(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.rnn = nn.RNN(emb_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out)

rnn_model = RNN(vocab_size, emb_size = 128, hidden_size = 128)  # create a instance of RNN model

In [15]:
print("RNN parameter count: ", count_params(rnn_model))  # parameter count

RNN parameter count:  39963


In [16]:
train(rnn_model, X, Y) # train the model

Epoch 1, Loss: 3.3625
Epoch 2, Loss: 3.2621
Epoch 3, Loss: 3.1679
Epoch 4, Loss: 3.0791
Epoch 5, Loss: 2.9952
Epoch 6, Loss: 2.9154
Epoch 7, Loss: 2.8392
Epoch 8, Loss: 2.7662
Epoch 9, Loss: 2.6963
Epoch 10, Loss: 2.6298
Epoch 11, Loss: 2.5675
Epoch 12, Loss: 2.5105
Epoch 13, Loss: 2.4604
Epoch 14, Loss: 2.4183
Epoch 15, Loss: 2.3845
Epoch 16, Loss: 2.3573
Epoch 17, Loss: 2.3339
Epoch 18, Loss: 2.3119
Epoch 19, Loss: 2.2901
Epoch 20, Loss: 2.2689
Epoch 21, Loss: 2.2486
Epoch 22, Loss: 2.2297
Epoch 23, Loss: 2.2121
Epoch 24, Loss: 2.1954
Epoch 25, Loss: 2.1793
Epoch 26, Loss: 2.1638
Epoch 27, Loss: 2.1487
Epoch 28, Loss: 2.1342
Epoch 29, Loss: 2.1204
Epoch 30, Loss: 2.1073
Epoch 31, Loss: 2.0951
Epoch 32, Loss: 2.0836
Epoch 33, Loss: 2.0730
Epoch 34, Loss: 2.0632
Epoch 35, Loss: 2.0538
Epoch 36, Loss: 2.0447
Epoch 37, Loss: 2.0358
Epoch 38, Loss: 2.0272
Epoch 39, Loss: 2.0189
Epoch 40, Loss: 2.0109
Epoch 41, Loss: 2.0031
Epoch 42, Loss: 1.9955
Epoch 43, Loss: 1.9883
Epoch 44, Loss: 1.98

In [17]:
torch.save(rnn_model.state_dict(), 'rnn_model.pth') # save the model

### Bi-LSTM

In [19]:
# define bidirectional lstm 

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.lstm = nn.LSTM(
            emb_size, hidden_size,
            batch_first=True,
            bidirectional=True # makes the LSTM bi directional 
        )
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out)

bilstm_model = BiLSTM(vocab_size,emb_size = 128, hidden_size = 128) # create a instance of biLSTM

In [20]:
print("Bi-LSTM params:", count_params(bilstm_model)) # parameter count

Bi-LSTM params: 274587


In [21]:
train(bilstm_model, X, Y, epochs = 75) # train the model

Epoch 1, Loss: 3.3018
Epoch 2, Loss: 3.2211
Epoch 3, Loss: 3.1416
Epoch 4, Loss: 3.0613
Epoch 5, Loss: 2.9789
Epoch 6, Loss: 2.8938
Epoch 7, Loss: 2.8060
Epoch 8, Loss: 2.7168
Epoch 9, Loss: 2.6284
Epoch 10, Loss: 2.5443
Epoch 11, Loss: 2.4680
Epoch 12, Loss: 2.4020
Epoch 13, Loss: 2.3459
Epoch 14, Loss: 2.2960
Epoch 15, Loss: 2.2470
Epoch 16, Loss: 2.1958
Epoch 17, Loss: 2.1425
Epoch 18, Loss: 2.0899
Epoch 19, Loss: 2.0402
Epoch 20, Loss: 1.9937
Epoch 21, Loss: 1.9492
Epoch 22, Loss: 1.9058
Epoch 23, Loss: 1.8627
Epoch 24, Loss: 1.8201
Epoch 25, Loss: 1.7779
Epoch 26, Loss: 1.7366
Epoch 27, Loss: 1.6965
Epoch 28, Loss: 1.6574
Epoch 29, Loss: 1.6188
Epoch 30, Loss: 1.5804
Epoch 31, Loss: 1.5419
Epoch 32, Loss: 1.5032
Epoch 33, Loss: 1.4648
Epoch 34, Loss: 1.4268
Epoch 35, Loss: 1.3896
Epoch 36, Loss: 1.3529
Epoch 37, Loss: 1.3165
Epoch 38, Loss: 1.2803
Epoch 39, Loss: 1.2445
Epoch 40, Loss: 1.2091
Epoch 41, Loss: 1.1742
Epoch 42, Loss: 1.1398
Epoch 43, Loss: 1.1062
Epoch 44, Loss: 1.07

In [22]:
torch.save(bilstm_model.state_dict(), 'bilstm_model.pth') # save the model

### RNN with Attention

In [24]:
# define RNN with attention 

class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, emb_size=128, hidden_size=256):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.dropout = nn.Dropout(0.3)

        self.rnn = nn.RNN(
            input_size=emb_size,
            hidden_size=hidden_size,
            num_layers=2,              
            nonlinearity='tanh',
            batch_first=True,
            dropout=0.3
        )

        self.attn = nn.Linear(hidden_size, hidden_size)  # attention

        
        self.norm = nn.LayerNorm(hidden_size) # normalize the values 

        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        B, T = x.size()
        
        emb = self.embedding(x)
        emb = self.dropout(emb)

        outputs, _ = self.rnn(emb)   # (B, T, H)
        outputs = self.norm(outputs)

        all_outputs = []

        for t in range(T):
            h_t = outputs[:, t:t+1, :]       # (B,1,H)

            past = outputs[:, :t+1, :]       # (B,t+1,H)

            scores = torch.bmm(
                h_t, self.attn(past).transpose(1, 2)
            )  # (B,1,t+1)

            weights = F.softmax(scores, dim=-1)

            context = torch.bmm(weights, past)  # (B,1,H)

            combined = torch.cat([h_t, context], dim=-1)  # (B,1,2H)

            out = self.fc(combined.squeeze(1))  # (B,V)

            all_outputs.append(out.unsqueeze(1))

        outputs = torch.cat(all_outputs, dim=1) # conatenate the tensors

        return outputs

attn_model = AttentionRNN(vocab_size,emb_size = 128, hidden_size = 256)

In [25]:
print("Attention RNN params:", count_params(attn_model))  # parameter count

Attention RNN params: 314011


In [26]:
train(attn_model, X, Y, epochs = 120) # train the model

Epoch 1, Loss: 3.4355
Epoch 2, Loss: 2.9038
Epoch 3, Loss: 2.6263
Epoch 4, Loss: 2.4654
Epoch 5, Loss: 2.3729
Epoch 6, Loss: 2.2888
Epoch 7, Loss: 2.2204
Epoch 8, Loss: 2.1719
Epoch 9, Loss: 2.1403
Epoch 10, Loss: 2.1086
Epoch 11, Loss: 2.0803
Epoch 12, Loss: 2.0594
Epoch 13, Loss: 2.0321
Epoch 14, Loss: 2.0202
Epoch 15, Loss: 1.9996
Epoch 16, Loss: 1.9858
Epoch 17, Loss: 1.9719
Epoch 18, Loss: 1.9599
Epoch 19, Loss: 1.9443
Epoch 20, Loss: 1.9333
Epoch 21, Loss: 1.9170
Epoch 22, Loss: 1.9014
Epoch 23, Loss: 1.8951
Epoch 24, Loss: 1.8855
Epoch 25, Loss: 1.8809
Epoch 26, Loss: 1.8670
Epoch 27, Loss: 1.8591
Epoch 28, Loss: 1.8483
Epoch 29, Loss: 1.8458
Epoch 30, Loss: 1.8320
Epoch 31, Loss: 1.8308
Epoch 32, Loss: 1.8158
Epoch 33, Loss: 1.8189
Epoch 34, Loss: 1.8153
Epoch 35, Loss: 1.8020
Epoch 36, Loss: 1.7963
Epoch 37, Loss: 1.7818
Epoch 38, Loss: 1.7741
Epoch 39, Loss: 1.7801
Epoch 40, Loss: 1.7674
Epoch 41, Loss: 1.7641
Epoch 42, Loss: 1.7578
Epoch 43, Loss: 1.7543
Epoch 44, Loss: 1.73

In [27]:
torch.save(attn_model.state_dict(), 'attn_model.pth') # save the model

## Inference and Evaluation

In [29]:
def generate_name(model, char2idx, idx2char, max_len=15, temperature=0.6, top_k=3):
    model.eval()

    SOS = char2idx["<SOS>"]
    EOS = char2idx["<EOS>"]
    PAD = char2idx.get("<PAD>", None)

    # better starting strategy to get realistic names
    start_chars = list("asrkmnpvtd")
    start_char = random.choice(start_chars) 

    seq = [SOS, char2idx.get(start_char, SOS)]
    generated = seq.copy()

    for _ in range(max_len):
        x = torch.tensor([seq])

        with torch.no_grad():
            logits = model(x)[0, -1]

        logits = logits / temperature

        logits[SOS] = -1e9
        if PAD is not None:
            logits[PAD] = -1e9
            
        for idx in set(generated):
            logits[idx] /= 1.5

        if len(generated) >= 2 and generated[-1] == generated[-2]:
            logits[generated[-1]] = -1e9
            
        probs = F.softmax(logits, dim=0)
        topk_probs, topk_idx = torch.topk(probs, top_k)

        topk_probs = topk_probs / topk_probs.sum()
        next_idx = topk_idx[torch.multinomial(topk_probs, 1)].item()

        if next_idx == EOS:
            break

        seq.append(next_idx)
        generated.append(next_idx)

    # convert to string (skip SOS)
    name = "".join(idx2char[i] for i in seq[1:])

    return name

In [30]:
def generate_batch(model, n=100):  # define function to generate samples 
    names = []
    for _ in range(n):
        name = generate_name(model, char2idx, idx2char)
        names.append(name)
    return names

In [31]:
# generating samples for each model

rnn_gen = generate_batch(rnn_model, 500)  
bilstm_gen = generate_batch(bilstm_model, 500)
attn_gen = generate_batch(attn_model, 500)

In [32]:
train_set = set(names)

def novelty(samples):
    new = [s for s in samples if s not in train_set]  
    return len(new) / len(samples) # computing novelty

def diversity(samples):
    return len(set(samples)) / len(samples)  # computing diversity

In [33]:
def evaluate(samples, name):
    print(f"{name}")
    print("Novelty:", novelty(samples)) 
    print("Diversity:", diversity(samples))
    print("-" * 30)


# compare models based on novelty and diversity

evaluate(rnn_gen, "RNN")
evaluate(bilstm_gen, "BiLSTM")
evaluate(attn_gen, "RNN + Attention")

RNN
Novelty: 0.854
Diversity: 0.662
------------------------------
BiLSTM
Novelty: 1.0
Diversity: 0.984
------------------------------
RNN + Attention
Novelty: 0.602
Diversity: 0.488
------------------------------


In [34]:
rnn_gen = generate_batch(rnn_model, 10)  
bilstm_gen = generate_batch(bilstm_model, 10)
attn_gen = generate_batch(attn_model, 10)

In [43]:
print(rnn_gen)

['nitesh', 'karan', 'navit', 'shani', 'malini', 'anshit', 'devin', 'viran', 'divyan', 'pradish']


In [44]:
print(bilstm_gen)

['surejandhi', 'pandhiteesh', 'viteranndh', 'derandhuna', 'prejant', 'aniretaya', 'noogolesh', 'niteryal', 'mokarulit', 'ragondepikk']


In [45]:
print(attn_gen)

['shrekan', 'kairavya', 'sarika', 'shivan', 'mahavin', 'triloka', 'sandhya', 'rajish', 'tanush', 'surendera']
